# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a workflow for loading and exploring the FAIR² Rangeland Management dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nPublished:", getattr(metadata, 'datePublished', None))
print("Keywords:", getattr(metadata, 'keywords', None))

## 2. Data Overview

Review available record sets, fields, and their unique `@id`s.

We'll enumerate all record sets defined in the Croissant schema, inspecting their `@id`, and show fields/columns for each. This enables downstream referencing _only by_ `@id`, as required.

In [ ]:
# Discover all record sets by @id
record_sets = list(dataset.record_sets)
print(f"There are {len(record_sets)} record set(s) in this dataset:")
for rs in record_sets:
    print(f"  - {rs['@id']} | name: {rs.get('name', None)}")

# For each record set, show their fields/columns (by @id)
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nFields/columns for record set {rs_id}:")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        print(f"  - {f.get('@id', str(f))}, name: {f.get('name', None)}, dataType: {f.get('dataType', None)}")

## 3. Data Extraction

Load data from the available record sets into DataFrames for analysis. Use the `@id` values discovered above for referencing record sets and fields. DataFrames are created for each record set.

**Note:** For reproducibility, we store all loaded DataFrames in a dictionary keyed by the record set's `@id`.

In [ ]:
# We collect all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set {record_set_id}")
        if not df.empty:
            print("Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load records for {record_set_id}:", e)

# For demonstration, pick the first (primary) record set if available
primary_record_set_id = record_set_ids[0] if record_set_ids else None
if primary_record_set_id and not dataframes[primary_record_set_id].empty:
    print(f"\nFirst rows of {primary_record_set_id}:")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Below, we process the loaded DataFrames for initial exploration. We filter on a numeric field, normalize its values, and, if possible, group by a categorical field.

**Before running,** manually inspect above which fields seem numeric, and which provide a meaningful group.

In [ ]:
# Replace with correct @ids from your overview above as needed:
record_set_id = primary_record_set_id
df = dataframes.get(record_set_id, pd.DataFrame())

# Heuristically select a numeric field (update manually as per your data):
numeric_field = None
for col in df.columns:
    # look for likely numeric fields
    if df[col].dtype.kind in {'i', 'u', 'f'} and not df[col].isnull().all():
        numeric_field = col
        break
if not numeric_field:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field = col
            break
        except Exception:
            continue

print('Numeric field selected:', numeric_field)
if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in {'i', 'u', 'f'} else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to heuristically pick a group field
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and df[col].nunique() < (len(df) / 2):
            group_field = col
            break
    print('Group field selected:', group_field)
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of a numeric field and a grouped barplot if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# If group_field exists, visualize group means
if 'group_field' in locals() and group_field and numeric_field:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_means.index, y=group_means.values)
    plt.xticks(rotation=60)
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- The dataset is structured via record sets, with all exploration referencing entities by their unique `@id`.
- Initial EDA gives insights into numeric variable distributions and possible categorical breakdowns.
- For more refined analyses, repeat the workflow for other record sets and fields as needed (using their `@id`).
- When publishing results or code, always reference dataset elements by `@id` to remain aligned with the Croissant schema best practices.

**For detailed semantic documentation and advanced schema navigation, see the Croissant schema at:**

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)